# Deep Learning Model for Image Classification

## Project Overview
This notebook implements a Convolutional Neural Network (CNN) for image classification using TensorFlow/Keras. The model is trained on the CIFAR-10 dataset to classify images into 10 different categories.

**Task**: Implement a Deep Learning Model for Image Classification using TensorFlow

**Deliverable**: A functional model with visualizations of results.

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

## 2. Load and Explore the CIFAR-10 Dataset

In [ ]:
# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Class names for CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Explore dataset structure
print("\n=== Dataset Information ===")
print(f"Training set shape: {x_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test set shape: {x_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Number of classes: {len(class_names)}")
print(f"Image shape: {x_train[0].shape}")
print(f"Data type: {x_train.dtype}")
print(f"Pixel value range: [{x_train.min()}, {x_train.max()}]")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Images from CIFAR-10 Dataset', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(x_train))
    ax.imshow(x_train[idx].astype('uint8'))
    ax.set_title(f'Class: {class_names[y_train[idx][0]]}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Class distribution
print("\n=== Class Distribution in Training Set ===")
unique, counts = np.unique(y_train, return_counts=True)
for cls_name, count in zip(class_names, counts):
    print(f"{cls_name}: {count} images")

## 3. Preprocess and Prepare Data

In [ ]:
# Normalize pixel values to [0, 1]
x_train_normalized = x_train.astype('float32') / 255.0
x_test_normalized = x_test.astype('float32') / 255.0

# Convert labels to one-hot encoding
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# Create validation split
from sklearn.model_selection import train_test_split
x_train_split, x_val, y_train_split, y_val = train_test_split(
    x_train_normalized, y_train_cat, test_size=0.2, random_state=42, stratify=y_train
)

print("\n=== Data Preprocessing Complete ===")
print(f"Training set shape (after normalization & split): {x_train_split.shape}")
print(f"Validation set shape: {x_val.shape}")
print(f"Test set shape (after normalization): {x_test_normalized.shape}")
print(f"Training labels shape: {y_train_split.shape}")
print(f"Pixel value range after normalization: [{x_train_normalized.min()}, {x_train_normalized.max()}]")

## 4. Build the CNN Model

In [ ]:
# Build Convolutional Neural Network Model
model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Flatten and Dense layers
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')  # Output layer for 10 classes
])

print("\n=== CNN Model Architecture ===")
model.summary()

## 5. Compile the Model

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")
print("\n=== Model Configuration ===")
print(f"Optimizer: Adam (learning_rate=0.001)")
print(f"Loss function: Categorical Crossentropy")
print(f"Metrics: Accuracy")

## 6. Train the Model

In [ ]:
# Set up callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train the model
print("Starting model training...")
epochs = 50
batch_size = 128

history = model.fit(
    x_train_split, y_train_split,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)

print("\nModel training completed!")

## 7. Evaluate Model Performance

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(x_test_normalized, y_test_cat, verbose=0)

print("\n=== Test Set Performance ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Get predictions
y_pred_probs = model.predict(x_test_normalized, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_test_true = np.argmax(y_test_cat, axis=1)

# Calculate metrics
print("\n=== Classification Report ===")
print(classification_report(y_test_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test_true, y_pred)
print("\n=== Confusion Matrix ===")
print(cm)

## 8. Visualize Results and Predictions

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize correct predictions
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Correct Predictions - Test Set', fontsize=14, fontweight='bold')

correct_indices = np.where(y_pred == y_test_true)[0]
selected_correct = np.random.choice(correct_indices, 10, replace=False)

for idx, ax in enumerate(axes.flat):
    test_idx = selected_correct[idx]
    ax.imshow(x_test_normalized[test_idx])
    ax.set_title(f'True: {class_names[y_test_true[test_idx]]}\nPred: {class_names[y_pred[test_idx]]}',
                fontsize=9, color='green')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize misclassified predictions
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Misclassified Predictions - Test Set', fontsize=14, fontweight='bold')

misclassified_indices = np.where(y_pred != y_test_true)[0]
selected_misclassified = np.random.choice(misclassified_indices, 10, replace=False)

for idx, ax in enumerate(axes.flat):
    test_idx = selected_misclassified[idx]
    ax.imshow(x_test_normalized[test_idx])
    ax.set_title(f'True: {class_names[y_test_true[test_idx]]}\nPred: {class_names[y_pred[test_idx]]}',
                fontsize=9, color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"\nTotal Misclassified Images: {len(misclassified_indices)} out of {len(y_test_true)}")
print(f"Misclassification Rate: {len(misclassified_indices)/len(y_test_true)*100:.2f}%")

In [ ]:
# Model summary and insights
print("\n" + "="*60)
print("DEEP LEARNING MODEL - FINAL SUMMARY".center(60))
print("="*60)
print(f"\nDataset: CIFAR-10 (Image Classification)")
print(f"Total Training Samples: {len(x_train_split)}")
print(f"Total Validation Samples: {len(x_val)}")
print(f"Total Test Samples: {len(x_test_normalized)}")
print(f"Number of Classes: 10")
print(f"Image Size: 32x32 RGB")

print(f"\n--- Model Architecture ---")
print(f"Type: Convolutional Neural Network (CNN)")
print(f"Total Parameters: {model.count_params():,}")
print(f"Trainable Parameters: {sum([tf.keras.backend.count_params(w) for w in model.trainable_weights]):,}")

print(f"\n--- Training Configuration ---")
print(f"Optimizer: Adam (learning_rate=0.001)")
print(f"Loss Function: Categorical Crossentropy")
print(f"Batch Size: 128")
print(f"Epochs Trained: {len(history.history['loss'])}")
print(f"Early Stopping: Enabled (patience=10)")

print(f"\n--- Performance Metrics ---")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print(f"Best Validation Accuracy: {max(history.history['val_accuracy'])*100:.2f}%")
print(f"Best Validation Loss: {min(history.history['val_loss']):.4f}")

print("\n" + "="*60)
print("Model training and evaluation completed successfully!".center(60))
print("="*60)